# STAGE 2: VQA with Qwen2-VL-2B on ChartQA
This stage covers the fine-tuning Qwen2-VL-2B on a subset of the ChartQA dataset using QLoRA, including dataset preprocessing, training, and evaluation.

## Plan Overview
**Phase 1:** Setup
1. Load ChartQA dataset: training, validation, and test sets
2. Inspect examples: visualize charts and read associated questions/answers

**Phase 2:** Fine-tuning with QLoRA
1. Load the pretrained base model (Qwen2-VL-2B-Instruct)
2. Prepare training data:
    - Convert each question-image pair into the Qwen2-VL chat format
    - Tokenize inputs and mask non-answer tokens for supervised fine-tuning
3. Precompute and cache images used for training (reduces training-time)
4. Set up QLoRA training configuration
5. Fine-tune the model:
    - Train on 2000 examples for 2 epochs
    - Monitor training loss
6. Save and upload the fine-tuned model:
    - Push the model (adaptor) and processor to the HuggingFace Hub

**Phase 3:** Evaluation
1. Load the base model
    - Evaluate the unmodified Qwen2-VL on a part of the ChartQA test set
2. Fine-tuned model (from Hugging Face Hub):
    - Merge LoRA weights with the base model for inference
3. Make predictions on the same test set with both models
4. Calculate metrics for both
    - Exact match accuracy,
    - Relaxed match accuracy,
    - Numeric accuracy
5. Compare base vs fine-tuned side-by-side
    - Show improvements in accuracy
6. Analyze errors

**Phase 4:** Combine Stage 1 and Stage 2 into the full pipeline (in a separate notebook)

## **PHASE 1: SET-UP**
### Load the Dataset
We load the ChartQA dataset from Hugging Face and inspect its size, as well as some examples.

In [ ]:
from datasets import load_dataset

ds = load_dataset("HuggingFaceM4/ChartQA")

train_ds = ds['train']
val_ds = ds['val']
test_ds = ds['test']

In [ ]:
print(f"Dataset size: \n train: {len(train_ds)} \n val: {len(val_ds)} \n test: {len(test_ds)}")
print(f"\nKeys: {train_ds[0].keys()}")
print(f"\nFirst example:")
print(train_ds[0])

### Display several random examples from the dataset

In [ ]:
import random
import matplotlib.pyplot as plt

for i in random.sample(range(len(train_ds)), 5):
    ex = train_ds[i]
    plt.imshow(ex["image"])
    plt.axis("off")
    print("Q:", ex["query"])
    print("A:", ex["label"])
    plt.show()

### Substitute "token" with your actual Hugging Face token!!!

In [ ]:
from huggingface_hub import login

# Login with your HF token
login(token="token") # PUT YOUR HUGGING FACE TOKEN HERE!!!!

## Phase 2: Fine-Tuning
### Model Set-up and QLoRA configuration
We use Qwen2-VL-2B-Instruct, which is a 2 billion parameter vision-language model. We chose this model because it's designed for visual question answering and chart understanding tasks, which aligns exactly with our goal.

QLoRA (Quantized Low-Rank Adaptation):
- Adds small trainable adapter layers (LoRA) while keeping the base model frozen, i.e. it adds lightweight, low-rank matrices to the original model instead of retraining the entire model
- Uses 4-bit quantization to reduce memory usage (very useful in our case due to GPU limitation)

Configuration choices:
- Rank r=8, alpha=16: Controls adapter capacity (higher = more expressive but slower). r - rank of the added low-rank decomposition matrices.
- Target modules: All attention and MLP projection layers for comprehensive adaptation
- 4-bit NF4 quantization: Optimal for neural network weights, preserves accuracy

This setup allows us to fine-tune a 2B parameter model on a single GPU with ~15GB VRAM and limited runtime.


In [ ]:
!pip install -q peft bitsandbytes accelerate qwen-vl-utils

from transformers import BitsAndBytesConfig
import torch


# Model setup (QLoRA-ready for Qwen2-VL)
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info

print("\n📥 Loading Qwen2-VL-2B with 4-bit quantization...")

# 4-bit config for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load processor + model
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id
    print(f"Set pad_token to eos_token (id: {processor.tokenizer.pad_token_id})")



model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False


# LoRA config for Qwen2-VL - targeting attention layers
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ Model ready!")


### Data Preprocessing & Caching

Preprocess ChartQA samples and save them to disk to avoid redundant computation during training.

For each sample, we:
1. Format the question-answer pair using Qwen2-VL's chat template
2. Process both image (into vision tokens) and text (into token IDs)
3. Mask the prompt portion of labels (so that the model is only trained on answers, not questions)
4. Save all preprocessed tensors to disk as .pt files

This preprocessing step runs once and significantly speeds up training by eliminating redundant image processing on every epoch. This approach is necessary with our limited GPU.

In [ ]:
# Precompute vision features for all images in a dataset
def precompute_vision_features_to_disk(hf_dataset, processor, out_dir, max_samples=None):
    os.makedirs(out_dir, exist_ok=True)

    dataset = hf_dataset.select(range(min(max_samples, len(hf_dataset)))) if max_samples else hf_dataset

    for i, item in enumerate(dataset):
        question = item["query"]
        answer = item["label"][0] if isinstance(item["label"], list) else item["label"]
        answer = str(answer).strip()

        messages = [
            {"role": "user", "content":[
                {"type": "image", "image": item["image"]},
                {"type": "text", "text": f"Answer this question about the chart: {question}"}
            ]},
            {"role": "assistant", "content":[{"type":"text","text": answer}]}
        ]

        text_full = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        text_prompt = processor.apply_chat_template(messages[:1], tokenize=False, add_generation_prompt=True)

        inputs = processor(
            text=[text_full],
            images=[item["image"]],
            padding=False,
            return_tensors="pt"
        )

        prompt_ids = processor.tokenizer(text_prompt, return_tensors="pt", add_special_tokens=False)["input_ids"]
        prompt_length = prompt_ids.shape[1]
        labels = inputs["input_ids"][0].clone()
        labels[:prompt_length] = -100

        # DEBUG: Print shapes for first sample
        if i == 0:
            print(f"DEBUG - pixel_values shape: {inputs['pixel_values'].shape}")
            print(f"DEBUG - image_grid_thw shape: {inputs['image_grid_thw'].shape}")

        # Save - keep pixel_values in its original shape
        torch.save({
            "input_ids": inputs["input_ids"][0],
            "attention_mask": inputs["attention_mask"][0],
            "labels": labels,
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "image_grid_thw": inputs["image_grid_thw"][0]
        }, os.path.join(out_dir, f"{i}.pt"))

        if (i+1) % 100 == 0:
            print(f"Saved {i+1}/{len(dataset)} files")
    print("Done saving precomputed features.")

### Data Collator

Handle batching and padding for Qwen2-VL's:
- Pad text sequences to the same length within each batch
- Concatenate vision patches from different images
- Uses image_grid_thw to tell the model which patches belong to which image

This approach handles variable-sized chart images efficiently while maintaining proper alignment between vision and language tokens.

In [ ]:
from torch.utils.data import Dataset
import torch
import os

class ChartQADatasetQwenDisk(Dataset):
    def __init__(self, folder):
        self.folder = folder
        self.files = sorted([f for f in os.listdir(folder) if f.endswith(".pt")])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = torch.load(os.path.join(self.folder, self.files[idx]))

        return {
            "input_ids": data["input_ids"],
            "attention_mask": data["attention_mask"],
            "labels": data["labels"],
            "pixel_values": data["pixel_values"],
            "image_grid_thw": data["image_grid_thw"]
        }



# Create cache once
import os

# Only precompute if cache doesn't exist
if not os.path.exists("train_cache") or len(os.listdir("train_cache")) == 0:
    print("Creating train cache...")
    precompute_vision_features_to_disk(train_ds, processor, "train_cache", max_samples=2000)
else:
    print("Train cache already exists, skipping preprocessing")

if not os.path.exists("val_cache") or len(os.listdir("val_cache")) == 0:
    print("Creating val cache...")
    precompute_vision_features_to_disk(val_ds, processor, "val_cache", max_samples=200)
else:
    print("Val cache already exists, skipping preprocessing")


# Load datasets from disk
train_dataset = ChartQADatasetQwenDisk("train_cache")
val_dataset = ChartQADatasetQwenDisk("val_cache")

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))


# Quick verification
sample = train_dataset[0]
print(f"input_ids shape: {sample['input_ids'].shape}")
print(f"labels shape: {sample['labels'].shape}")
print(f"pixel_values shape: {sample['pixel_values'].shape}")
print(f"image_grid_thw shape: {sample['image_grid_thw'].shape}")
print(f"Non-masked labels: {(sample['labels'] != -100).sum().item()}")

In [ ]:
from dataclasses import dataclass
from typing import Any, List, Dict
import torch

@dataclass
class Qwen2VLDataCollator:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # Pad text sequences
        input_ids = torch.nn.utils.rnn.pad_sequence(
            [f['input_ids'] for f in features],
            batch_first=True,
            padding_value=self.processor.tokenizer.pad_token_id
        )
        attention_mask = torch.nn.utils.rnn.pad_sequence(
            [f['attention_mask'] for f in features],
            batch_first=True,
            padding_value=0
        )
        labels = torch.nn.utils.rnn.pad_sequence(
            [f['labels'] for f in features],
            batch_first=True,
            padding_value=-100
        )

        # This is how Qwen2-VL handles batches with different image sizes
        pixel_values = torch.cat([f['pixel_values'] for f in features], dim=0)

        # Stack image_grid_thw - this tells the model where each image's patches are
        image_grid_thw = torch.stack([f['image_grid_thw'] for f in features])

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "pixel_values": pixel_values,  # [total_patches_in_batch, 1176]
            "image_grid_thw": image_grid_thw  # [batch_size, 3]
        }

# Recreate the collator
data_collator = Qwen2VLDataCollator(processor=processor)

### Training Configuration

Key training hyperparameters:
- 2 epochs with effective batch size of 16
- Learning rate: 1e-4 with 5% warmup for stable convergence
- Evaluation every 100 steps to monitor overfitting
- paged_adamw_8bit optimizer: memory-efficient for QLoRA training
- Saves best 2 checkpoints based on validation loss

There are 9,232,384 trainable parameters.

In [ ]:
# Cell 4: Training
model_name = "qwen2vl-2b-chartqa-qlora-2500-v3"

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=2,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=1e-4,
    warmup_ratio=0.05,

    logging_steps=50,
    eval_strategy="steps",
    eval_steps=100,

    save_steps=100,
    save_total_limit=2,

    bf16=False,
    fp16=True,

    push_to_hub=True,
    hub_model_id=model_name,
    hub_strategy="every_save",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    gradient_checkpointing=True, # Save memory
    optim="paged_adamw_8bit",

    remove_unused_columns=False,  # Keep all columns

    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("✅ Trainer ready!")
model.print_trainable_parameters()

### Fine-Tuning

Run the fine-tuning process. The trainer handles:
- Forward/backward passes through the model
- Gradient accumulation across batches
- Periodic evaluation and checkpoint saving

We did evaluation every 100 steps (2 times). Loss was around 7.3, with training and validation losses closely aligned.


In [ ]:
# Cell 6: Train
print("\n🚀 Starting training...")
print(f"Training on {len(train_dataset)} samples for {training_args.num_train_epochs} epochs")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print("="*70 + "\n")

trainer.train()

print("\n✅ Training complete!")

### Upload our Model to HuggingFace Hub
We upload the model to HuggingFace Hub, so that we can load it from there later.

In [ ]:
print("\n📤 Uploading model to HuggingFace Hub...")

# First, add the base model info to the adapter config
from peft import PeftModel

# Update adapter config with base model
model.config.base_model_name_or_path = "Qwen/Qwen2-VL-2B-Instruct"

# Save locally first
model.save_pretrained("./final_model")
processor.save_pretrained("./final_model")

# Now push with proper metadata
model.push_to_hub(
    model_name,
    commit_message="QLoRA fine-tuned Qwen2-VL-2B on ChartQA - 2000 v3"
)
processor.push_to_hub(model_name)

print("✅ Model uploaded successfully!")

# Phase 3: Evaluation
### Inference & Model Comparison

Load both the base Qwen2-VL-2B model and our fine-tuned version from HuggingFace Hub. The fine-tuned model consists of the base model + trained LoRA adapters, which we combine to make inference.

For one test example we:
1. Display the chart image and question
2. Make a prediction on both models with identical prompts
3. Look at their answers compared to the ground truth

We do this to directly check if there seems to be a difference between the predictions on random test examples.

Change TEST_INDEX to look at different examples from the test set.


In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

# Free up previous memory just in case
torch.cuda.empty_cache()

HF_USER = "vlaaa13"
ADAPTER_NAME = "qwen2vl-2b-chartqa-qlora-2500-v3"

### Load Base Model

In [ ]:
print("📥 Loading base model...")
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
base_processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
print("✅ Base model ready!")

### Load Fine-tuned Model

In [ ]:
print("📥 Loading fine-tuned model...")
finetuned_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
finetuned_model = PeftModel.from_pretrained(
    finetuned_model,
    f"{HF_USER}/{ADAPTER_NAME}"
)
finetuned_processor = AutoProcessor.from_pretrained(f"{HF_USER}/{ADAPTER_NAME}")
print("✅ Fine-tuned model ready!")

In [ ]:
# Run inference comparison (change TEST_INDEX to change the examples and re-run this cell)
TEST_INDEX = 10

from IPython.display import display

# Get example
test_sample = test_ds[TEST_INDEX]
image = test_sample["image"]
question = test_sample["query"]
ground_truth = test_sample["label"]

# Display
print("="*70)
print(f"📊 TEST EXAMPLE #{TEST_INDEX}")
print("="*70)
display(image)
print(f"\n❓ Question: {question}")
print(f"\n✅ Ground Truth: {ground_truth}")
print("\n" + "="*70)

# BASE MODEL inference
print("\n🤖 BASE MODEL Answer:")
print("-"*70)

messages = [
    {"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": f"Answer this question about the chart: {question}"}
    ]}
]

text = base_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = base_processor(
    text=[text],
    images=[image],
    return_tensors="pt",
    padding=True
).to(base_model.device)

with torch.no_grad():
    output_ids = base_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=base_processor.tokenizer.pad_token_id
    )

generated_ids = output_ids[0][inputs['input_ids'].shape[1]:]
base_answer = base_processor.decode(generated_ids, skip_special_tokens=True).strip()

print(f"{base_answer}")
print("-"*70)

# FINE-TUNED MODEL inference
print("\n🎯 FINE-TUNED MODEL Answer:")
print("-"*70)

text = finetuned_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = finetuned_processor(
    text=[text],
    images=[image],
    return_tensors="pt",
    padding=True
).to(finetuned_model.device)

with torch.no_grad():
    output_ids = finetuned_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=finetuned_processor.tokenizer.pad_token_id
    )

generated_ids = output_ids[0][inputs['input_ids'].shape[1]:]
finetuned_answer = finetuned_processor.decode(generated_ids, skip_special_tokens=True).strip()

print(f"{finetuned_answer}")
print("-"*70)

### Evaluation Function

Run evaluation on a dataset:
1. Iterate through test examples
2. Generate predictions using deterministic sampling (do_sample=False) for reproducibility
3. Extract only the model's answer
4. Display first 20 examples for qualitative analysis

Return predictions, ground truths, and questions for metric calculation. This function is reused for both base and fine-tuned model evaluation.


In [ ]:
# Evaluation function for Qwen2-VL
from tqdm import tqdm
import torch

def run_inference_qwen2vl(dataset, model, processor, device, max_samples=None, desc="Inference"):
    # Limit dataset if needed
    if max_samples:
        dataset = dataset.select(range(min(max_samples, len(dataset))))

    predictions = []
    ground_truths = []
    questions = []
    indices = list(range(len(dataset)))

    print(f"🔄 Running {desc} on {len(dataset)} examples...")

    for i in tqdm(range(len(dataset)), desc=desc):
        example = dataset[i]

        image = example['image']
        question = example['query']
        ground_truth = example['label'][0] if isinstance(example['label'], list) else example['label']
        ground_truth = str(ground_truth).strip()

        # Prepare messages in Qwen2-VL format
        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": f"Answer this question about the chart: {question}"}
            ]}
        ]

        # Process inputs
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(
            text=[text],
            images=[image],
            return_tensors="pt",
            padding=True
        ).to(device)

        # Generate answer
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,  # Deterministic for evaluation
                pad_token_id=processor.tokenizer.pad_token_id
            )

        # Decode only the generated part (skip input prompt)
        generated_ids = output_ids[0][inputs['input_ids'].shape[1]:]
        prediction = processor.decode(generated_ids, skip_special_tokens=True).strip()

        predictions.append(prediction)
        ground_truths.append(ground_truth)
        questions.append(question)

        # Print first 30 examples
        if i < 30:
            print(f"\n--- Example {i+1} ---")
            print(f"Q: {question}")
            print(f"Predicted: {prediction}")
            print(f"Ground Truth: {ground_truth}")

    return predictions, ground_truths, questions, indices

### Calculate metrics function
Function that calculates various accuracy metrics. It will also be used on the test data with the base model, as well as after fune-tuning.

**Exact accuracy:**
- Convert both prediction and ground truth to lowercase and strip extra spaces.
- Only exact matches are counted as correct.

**Relaxed accuracy:**
- Exact matches OR numerically close predictions.
- Useful when LLMs output extra words around the answer.

**Numeric accuracy:**
- Only applies if the ground truth is numeric.
- Checks whether the predicted value is numerically close to the ground truth within a 5% relative tolerance.
- Correctly handles percentages and decimals as equivalent (e.g., "0.62", "62", "62%" are treated as matching). This flexible numeric matching accounts for models outputting percentages in different formats.




In [ ]:
import re

def is_numeric_answer(text):
    if text is None:
        return False
    text = text.strip().replace("%", "")
    return bool(re.match(r"^-?\d*\.?\d+$", text))

def numeric_close(pred, gt, tolerance=0.05):
    try:
        p = float(pred.replace("%", ""))
        g = float(gt.replace("%", ""))

        # Direct match
        if g != 0 and abs(p - g) / abs(g) <= tolerance:
            return True

        # Percentage/decimal match: 0.62 == 62
        if g != 0 and abs(p * 100 - g) / abs(g) <= tolerance:
            return True

        # Reverse percentage: 62 == 0.62
        if g != 0 and abs(p - g * 100) / abs(g * 100) <= tolerance:
            return True

        # If ground truth is zero
        if g == 0 and abs(p) < 1e-9:
            return True

        return False
    except:
        return False

def evaluate_chartqa_predictions(predictions, answers):
    """
    predictions: list of model outputs (strings)
    answers: list of {"answer": "..."}
    """
    total = len(predictions)

    exact_matches = 0
    relaxed_matches = 0
    numeric_matches = 0
    numeric_total = 0

    for pred, gt_dict in zip(predictions, answers):
        gt = gt_dict["answer"]

        pred_norm = pred.strip().lower()
        gt_norm = gt.strip().lower()

        # 1️⃣ Exact match
        if pred_norm == gt_norm:
            exact_matches += 1

        # 2️⃣ Relaxed match (lowercase or numeric-close)
        relaxed_ok = False
        if pred_norm == gt_norm:
            relaxed_ok = True
        elif is_numeric_answer(pred_norm) and is_numeric_answer(gt_norm):
            if numeric_close(pred_norm, gt_norm):
                relaxed_ok = True
        if relaxed_ok:
            relaxed_matches += 1

        # 3️⃣ Numeric accuracy (percentage-safe)
        if is_numeric_answer(gt_norm):
            numeric_total += 1
            if is_numeric_answer(pred_norm) and numeric_close(pred_norm, gt_norm):
                numeric_matches += 1

    return {
        "total_questions": total,
        "numeric_questions": numeric_total,
        "exact_match_accuracy": 100 * exact_matches / total,
        "relaxed_accuracy": 100 * relaxed_matches / total,
        "numeric_accuracy": 100 * numeric_matches / numeric_total if numeric_total > 0 else 0,
    }

## Evaluate Base Model
We evaluated the pretrained Qwen2-VL-2B on 200 ChartQA test examples without any fine-tuning.

Results:
- Exact Match Accuracy: 41.00%
- Relaxed Accuracy: 46.00%
- Numeric Accuracy: 38.81%

The base model shows reasonable chart understanding capabilities out-of-the-box, correctly answering simple counting and direct value extraction questions. However, it struggles with complex calculations (averages, differences) and reasoning tasks. As can be seen from the example questions, the format is also often not what we want, even if the answer is contained in the prediction.

In [ ]:
# Evaluate Base model
print("="*70)
print("🤖 EVALUATING BASE MODEL")
print("="*70)

base_preds, base_gts, base_qs, base_idx = run_inference_qwen2vl(
    test_ds,
    base_model,
    base_processor,
    base_model.device,
    max_samples= 200,  # Change the number samples if needed
    desc="Base Model"
)

base_metrics = evaluate_chartqa_predictions(
    base_preds, [{"answer": x} for x in base_gts]
)

print("\n" + "="*70)
print("📊 BASE MODEL RESULTS")
print("="*70)
print(f"Total Questions: {base_metrics['total_questions']}")
print(f"Numeric Questions: {base_metrics['numeric_questions']}")
print(f"\nExact Match Accuracy: {base_metrics['exact_match_accuracy']:.2f}%")
print(f"Relaxed Accuracy: {base_metrics['relaxed_accuracy']:.2f}%")
print(f"Numeric Accuracy: {base_metrics['numeric_accuracy']:.2f}%")
print("="*70)

## Evaluate Fine-tunde Model
Now we evaluated our fine-tuned model (trained on 2000 ChartQA examples for 2 epochs) on the same 200 test samples.

Results:
- Exact Match: 45.50% (+4.50% improvement)
- Relaxed Match: 56.00% (+10.00% improvement)
- Numeric Accuracy: 50.75% (+11.94% improvement)

The fine-tuned model shows consistent improvements across all metrics. Notable improvements include:
- Better format consistency (outputs concise numerical answers rather than full sentences)
- Improved yes/no question handling
- More accurate percentage understanding

Challenges remaining:
- Complex calculations still problematic (averaging, multi-step reasoning)
- Some counting errors on dense charts
- Visual reasoning tasks need more training data

In [ ]:
# Evaluate FINE-TUNED model
print("\n" + "="*70)
print("🎯 EVALUATING FINE-TUNED MODEL")
print("="*70)

finetuned_preds, finetuned_gts, finetuned_qs, finetuned_idx = run_inference_qwen2vl(
    test_ds,
    finetuned_model,
    finetuned_processor,
    finetuned_model.device,
    max_samples= 200,  # Same number as base model
    desc="Fine-tuned Model"
)

finetuned_metrics = evaluate_chartqa_predictions(
    finetuned_preds,
    [{"answer": x} for x in finetuned_gts]
)

print("\n" + "="*70)
print("📊 FINE-TUNED MODEL RESULTS")
print("="*70)
print(f"Total Questions: {finetuned_metrics['total_questions']}")
print(f"Numeric Questions: {finetuned_metrics['numeric_questions']}")
print(f"\nExact Match Accuracy: {finetuned_metrics['exact_match_accuracy']:.2f}%")
print(f"Relaxed Accuracy: {finetuned_metrics['relaxed_accuracy']:.2f}%")
print(f"Numeric Accuracy: {finetuned_metrics['numeric_accuracy']:.2f}%")
print("="*70)

In [ ]:
# Compare results
print("\n" + "="*70)
print("🔍 COMPARISON")
print("="*70)

print(f"\n{'Metric':<25} {'Base Model':<15} {'Fine-tuned':<15} {'Improvement':<15}")
print("-"*70)

for metric_name, display_name in [
    ('exact_match_accuracy', 'Exact Match'),
    ('relaxed_accuracy', 'Relaxed Match'),
    ('numeric_accuracy', 'Numeric')
]:
    base_val = base_metrics[metric_name]
    ft_val = finetuned_metrics[metric_name]
    improvement = ft_val - base_val

    print(f"{display_name:<25} {base_val:>6.2f}%        {ft_val:>6.2f}%        {improvement:>+6.2f}%")

print("="*70)

## **EVALUATION SUMMARY**

Compared base Qwen2-VL-2B against our fine-tuned version on 200 ChartQA test examples.

Key Findings:
- Consistent, modest improvements: +4.5% exact match shows the model learned ChartQA-specific patterns
- Format adaptation was successful: Fine-tuning taught the model to output concise answers (e.g., "3" instead of "There are three bars")
- Limited by training data: With only 2000 training examples and 2 epochs, gains are moderate as expected

The improvements validate our QLoRA approach works, but suggest potential for larger gains with:
- More training data (5000+ samples)
- Additional training epochs (4-6 epochs)
- Lower learning rate for better convergence (training loss plateaued at ~7.3), but would have probably improved with more training data and epochs
- Our results show that our fine-tuning worked successfully, given the limited resources (GPU and time constraints). This shows that using the same technique but with more computing power and more time, we would be able to achieve much better results and accuracy for our type of charts and questions.